[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/koebe/koebe.ipynb)

# Koebe Polyhedra

Every triangulation of the sphere is the pattern of a convex polyhedron whose edges all touch the unit sphere. Its polar dual then touches the sphere at the same points, each dual edge crossing its edge there at right angles: the best-shaped pair of a mesh and its dual there is, every dual vertex inside its face and every dual edge positive. The algebra here has the unit sphere as its metric, x, y and z squaring to one and w to minus one, and in it a circle on the sphere is a single vector, two circles touching is one inner product, and the maps that keep the sphere are its rotors.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np
from scipy.optimize import least_squares, minimize
from scipy.spatial import ConvexHull

from numga import NumpyContext
from examples.animation import save_figure, show_animation
from examples.surfaces.koebe import render
from examples.surfaces.koebe.core import ga   # x+y+z+w-: the unit sphere as the metric, shared with the figures

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()
Rotor = ga.gatype.rotor()

# The number of points, how far their directions are nudged off an even spread, and its seed; the spread and
# seed of the random distances whose hull gives the triangulation, every point on it.
points_count = 32
jitter = 0.03
direction_seed = 0
mesh_spread, mesh_seed = 0.06, 57
# The boosts the centred polyhedron is carried through: their largest rapidity, the tilt of the direction
# they turn about the z axis through, the frames of one turn, and the duration of each.
rapidity = 0.6
tilt = 0.4
frames_count = 48
duration_ms = 80


def hull(points: Vector) -> np.ndarray:
    """The faces of the convex hull of points around the centre, each turned so that the volume of its
    vertices turns as x, y, z do: counter-clockwise from outside."""
    faces = ConvexHull(render.components(points)[..., :3]).simplices          # [faces, 3]
    a, b, c = (points[faces[:, corner]] for corner in range(3))               # [faces] Vector each
    turning = render.scalars((a ^ b ^ c) * mv.xyz.reverse()) > 0              # [faces]
    return np.where(turning[:, None], faces, faces[:, [0, 2, 1]])


# Directions along a Fibonacci spiral, each nudged a little at random, and the hull of the points at random
# distances along them: a triangulation of the sphere with triangles of uneven shape.
turns = np.arange(points_count) + 0.5
polar, azimuth = np.arccos(1 - 2 * turns / points_count), np.pi * (1 + 5**0.5) * turns
spiral = np.stack([np.cos(azimuth) * np.sin(polar), np.sin(azimuth) * np.sin(polar), np.cos(polar)], axis=-1)   # [points, 3]
nudged = spiral + jitter * np.random.default_rng(direction_seed).normal(size=(points_count, 3))
directions = mv("x y z", nudged / np.linalg.norm(nudged, axis=-1, keepdims=True))   # [points] Vector
faces = hull(directions * np.exp(mesh_spread * np.random.default_rng(mesh_seed).normal(size=points_count)))
edges = np.unique(np.sort(np.concatenate([faces[:, [0, 1]], faces[:, [1, 2]], faces[:, [2, 0]]]), axis=-1), axis=0)

## 1. Circles that touch along every edge

A point x of the sphere is the null vector `x + w`, and the circle where the plane `n | x == s` cuts the sphere is the vector `n + s * w`, scaled to square to one: a point lies on it exactly when their inner product is zero. Two circles touch from outside exactly when their inner product is minus one, and their sum is then the null vector of the point where they touch. Read as a point, a circle's vector is the apex of the cone that touches the sphere along it, so circles touching along every edge are the vertices of a polyhedron whose every edge touches the sphere where its two circles do. The circle of a face, through its three touching points, is orthogonal to its vertices' circles, the dual of their wedge, and its apex is a vertex of the dual polyhedron. Finding the circles is one least-squares solve, from caps around the triangulation's directions: every circle squaring to one and every edge's inner product minus one, which fixes them up to the six dimensions of maps that keep the sphere.

In [ ]:
def caps(directions: Vector, radii: np.ndarray) -> Vector:
    """The circles on the sphere around unit directions, at angular radii: the plane at the cosine along each
    direction, `direction + cos(radius) * w`, scaled to square to one."""
    return (directions + mv.w * np.cos(radii)) * (1 / np.sin(radii))          # [...] Vector


def touching(circles: Vector, edges: np.ndarray) -> Vector:
    """The null vector of the point where the circles at the ends of each edge touch: their sum."""
    return circles[..., edges[:, 0]] + circles[..., edges[:, 1]]               # [..., edges] Vector


def face_circles(circles: Vector, faces: np.ndarray) -> Vector:
    """The circle of each face, through the points where its vertices' circles touch: orthogonal to all three,
    the dual of their wedge."""
    return (circles[..., faces[:, 0]] ^ circles[..., faces[:, 1]] ^ circles[..., faces[:, 2]]).dual()   # [..., faces] Vector


def apexes(circles: Vector) -> Vector:
    """The apex of the cone touching the sphere along each circle, as a point at unit weight along w."""
    return circles / -(circles | mv.w)                                         # [...] Vector


def koebe(start: Vector, edges: np.ndarray) -> Vector:
    """The circles nearest the start, by a least-squares solve, that square to one and touch from outside
    along every edge: inner product minus one."""
    basis = mv("x y z w", np.eye(4))                                           # [4] Vector
    gram = render.scalars(basis[:, None] | basis[None, :])                     # [4, 4] the metric
    count = start.shape[-1]

    def residual(flat: np.ndarray) -> np.ndarray:
        c = flat.reshape(count, 4)
        return np.concatenate([np.einsum("ni,ij,nj->n", c, gram, c) - 1,
                               np.einsum("ei,ij,ej->e", c[edges[:, 0]], gram, c[edges[:, 1]]) + 1])

    def jacobian(flat: np.ndarray) -> np.ndarray:
        c = flat.reshape(count, 4)
        rows = np.zeros((count + len(edges), count, 4))
        rows[np.arange(count), np.arange(count)] = 2 * c @ gram
        rows[count + np.arange(len(edges)), edges[:, 0]] = c[edges[:, 1]] @ gram
        rows[count + np.arange(len(edges)), edges[:, 1]] = c[edges[:, 0]] @ gram
        return rows.reshape(len(rows), -1)

    solved = least_squares(residual, render.components(start).reshape(-1), jac=jacobian, method="trf", xtol=1e-15, ftol=1e-15, gtol=1e-15).x
    return mv("x y z w", solved.reshape(count, 4))                             # [vertices] Vector


# Caps around the directions, half as wide as the mean angle to their neighbours.
angles = np.arccos(np.clip(render.scalars(directions[edges[:, 0]] | directions[edges[:, 1]]), -1, 1))   # [edges]
radii = np.bincount(edges.reshape(-1), np.repeat(angles, 2)) / np.bincount(edges.reshape(-1)) / 2   # [points]
start = caps(directions, radii)                                                # [points] Vector
circles = koebe(start, edges)                                                  # [points] Vector

print(f"inner products across the edges at the start, from {render.scalars(start[edges[:, 0]] | start[edges[:, 1]]).min():.2f} "
      f"to {render.scalars(start[edges[:, 0]] | start[edges[:, 1]]).max():.2f}; solved, all within "
      f"{np.abs(render.scalars(circles[edges[:, 0]] | circles[edges[:, 1]]) + 1).max():.0e} of minus one")

## 2. The maps that keep the sphere

The circles are fixed only up to the maps that keep the sphere, and those are the rotors of this algebra: rotations, and boosts, the rotors of a plane through w, which crowd the points of the sphere towards one direction, shrinking the circles there and growing those opposite. Every rotor keeps every inner product, so every boosted pattern still touches along every edge and every boosted polyhedron is still a Koebe polyhedron. The centred one has its touching points balanced about the centre: the unit timelike vector minimising the summed logarithms of minus its inner products with them is the centre from which they balance, and the rotor carrying it onto w centres the pattern. The animation carries the centred pattern through boosts whose direction turns once about the z axis.

In the language of circle packings, the vertex circles are a packing whose tangency pattern is the triangulation and the face circles its dual packing, which the Koebe–Andreev–Thurston theorem says exists and is unique up to Möbius transformations of the sphere.

In [ ]:
def centring(centre: Vector) -> Rotor:
    """The rotor carrying a unit timelike centre onto w: a boost along the centre's spatial part."""
    return (1 - mv.w * centre).normalized()


def boost(direction: Vector, rapidity: np.ndarray) -> Rotor:
    """The boost along a spatial unit direction by a rapidity."""
    return ((direction ^ mv.w) * (rapidity / 2)).exp()


def centre(points: Vector) -> Vector:
    """The unit timelike vector that minimises the summed logarithms of minus its inner products with the
    null vectors of points of the sphere: the centre seen from which they balance."""
    null = render.components(points)                                           # [points, 4]
    basis = mv("x y z w", np.eye(4))                                           # [4] Vector
    gram = render.scalars(basis[:, None] | basis[None, :])                     # [4, 4] the metric

    def timelike(spatial: np.ndarray) -> np.ndarray:
        return np.concatenate([spatial, [np.sqrt(1 + spatial @ spatial)]])

    spatial = minimize(lambda spatial: np.log(-(null @ gram @ timelike(spatial))).sum(), np.zeros(3), method="BFGS").x
    return mv("x y z w", timelike(spatial))                                    # [] Vector


centred = centring(centre(touching(circles, edges))) >> circles                # [points] Vector
save_figure(render.draw(centred, faces, edges), "koebe")

turn = np.linspace(0, 2 * np.pi, frames_count, endpoint=False)                 # [frames]
boost_directions = mv("x y z", np.stack([np.cos(turn), np.sin(turn), np.full_like(turn, tilt)], axis=-1)).normalized()   # [frames] Vector
boosted = boost(boost_directions, rapidity * np.sin(turn) ** 2)[:, None] >> centred   # [frames, points] Vector
show_animation(render.animate(boosted, faces, edges), "koebe_boost", duration_ms)

In [ ]:
# checks
# Every circle squares to one, and the circles at the ends of every edge touch from outside, under every boost.
assert np.abs(render.scalars(boosted | boosted) - 1).max() < 1e-10
assert np.abs(render.scalars(boosted[:, edges[:, 0]] | boosted[:, edges[:, 1]]) + 1).max() < 1e-10
# Every edge of the polyhedron and of its dual touches the sphere: its line's distance from the centre is one.
assert np.abs(render.edge_distances(apexes(boosted), edges) - 1).max() < 1e-10
assert np.abs(render.edge_distances(apexes(face_circles(boosted, faces)), render.beside(faces, edges)) - 1).max() < 1e-10
# Centred, the touching points balance about the centre.
assert np.linalg.norm(render.coordinates(touching(centred, edges)).mean(axis=0)) < 1e-4